In [1]:
## Build task_events.tsv for a single task

# Given a `task_label` and the acquisition
# output folder containing `logs.parquet`, emit a BIDS-style `task_events.tsv`
# with columns `onset`, `duration`, `trial_type`. Onsets are seconds relative
# to the first `time_ns` in `logs.parquet` (the recording start).

from pathlib import Path
import pandas as pd

output_folder = Path("/Users/clairenastaskin/data/sub-Forest001/2026-05-06/pathfinder.forest/events/fingers_motor/069fbd13-1983-7f62-8000-891686a18b10/")
ref_table = "task_table.tsv"
logs_path = output_folder / "logs.parquet"
task_table_path = output_folder / ref_table
assert logs_path.exists(), f"logs.parquet not found at {logs_path}"
assert task_table_path.exists(), f"{ref_table} not found at {task_table_path}"
print(f"output_folder   = {output_folder}")

output_folder   = /Users/clairenastaskin/data/sub-Forest001/2026-05-06/pathfinder.forest/events/fingers_motor/069fbd13-1983-7f62-8000-891686a18b10


In [2]:
### Load recording reference time from `logs.parquet`

# The first `time_ns` in `logs.parquet` is the wall-clock timestamp of the
# first image of the recording; all event onsets are measured relative to it.

logs_df = pd.read_parquet(logs_path, columns=["time_ns"])
recording_start_ns = int(logs_df["time_ns"].min())
tr_seconds = float(logs_df["time_ns"].sort_values().diff().mean()) / 1e9
print(f"recording_start_ns = {recording_start_ns}")
print(f"Tr (avg inter-image interval) = {tr_seconds:.6f} s")

recording_start_ns = 1778110812194442632
Tr (avg inter-image interval) = 2.001968 s


In [3]:
### Build and write `task_events.tsv`

task_table = pd.read_csv(task_table_path, sep="\t")
required_cols = {"screen_name", "start_time_ns", "duration"}
missing = required_cols - set(task_table.columns)
assert not missing, f"{ref_table} missing required columns: {missing}"

# Set the recording origin to the middle of the first TR.
onset_seconds = (task_table["start_time_ns"].astype("int64") - recording_start_ns) / 1e9 + tr_seconds / 2

events = pd.DataFrame(
    {
        "onset": onset_seconds,
        "duration": task_table["duration"].astype(float),
        "trial_type": task_table["screen_name"],
    }
)

In [ ]:
### OPTIONAL:  Keep only `words*` trials and rename them to `words`

events = events[events["trial_type"].str.startswith("video")].copy()
events["trial_type"] = "fingers"

events_path = output_folder / "task_events.tsv"
events.to_csv(events_path, sep="\t", index=False, float_format="%.6f")
print(f"wrote {events_path}")
print(events.to_string(index=False))

wrote /Users/clairenastaskin/data/sub-Forest001/2026-05-06/pathfinder.forest/events/fingers_motor/069fbd13-1983-7f62-8000-891686a18b10/task_events.tsv
     onset  duration trial_type
 11.738633 12.361911    fingers
 35.881726 12.249728    fingers
 59.909937 12.144803    fingers
 83.834934 12.200268    fingers
107.816790 12.223003    fingers
131.818847 12.191652    fingers
155.787115 12.144539    fingers
179.959709 12.209939    fingers
203.952535 12.251319    fingers
227.987056 12.286927    fingers
251.802594 12.290633    fingers
275.872380 12.205694    fingers
299.856350 12.209837    fingers
323.845701 12.289960    fingers
347.916697 12.291552    fingers
